# Ingest drivers.json file
1. Read the file using spark dataframe reader API
1. Define and enforce schema (preserve the nested structure)
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
%run ../00-common-Config/02-helper-function

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
source_file=f"{landing_path}/{v_batch_id}/drivers.json"
table_name=f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType,DateType
name_struct = StructType([
   StructField('givenName', StringType()),
   StructField('familyName', StringType())
 ])
driver_schema=StructType([
   StructField('driverId', StringType()),
   StructField('name', name_struct),
   StructField('dateOfBirth', DateType()),
   StructField('nationality', StringType()),
   StructField('url', StringType())
 ])

In [0]:
driver_df=spark.read.format("json")\
    .schema(driver_schema)\
    .option('mode','FAIL_FAST')\
    .load(source_file)

driver_df.show()

In [0]:
driver_df=add_ingestion_metadata(driver_df)


In [0]:
# driver_df.write.format("delta")\
#     .mode("overwrite")\
#     .saveAsTable(table_name)
# display(spark.sql(f"SELECT * FROM {table_name}"))

write_to_bronze(
    input_df=driver_df,
    target_table=table_name,
    batch_id=v_batch_id
)

In [0]:
display(spark.sql(f"SELECT * FROM {table_name}"))